# Delivery Exception & Late-Shipment Analysis

Dataset: DataCo Smart Supply Chain (Kaggle). Goal: find how often orders ship late, which shipping modes drive it, and build a prioritized exception queue for escalation.

## 1. Load data and prepare one row per order

In [2]:
import pandas as pd, numpy as np, zipfile, os
# Upload the Kaggle zip (or the CSV) to Colab first, then run this cell
for f in os.listdir("."):
    if f.endswith(".zip"):
        zipfile.ZipFile(f).extractall()
df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="latin-1")

real, sched = "Days for shipping (real)", "Days for shipment (scheduled)"
late, region, mode = "Late_delivery_risk", "Order Region", "Shipping Mode"
odate, qty, sales = "order date (DateOrders)", "Order Item Quantity", "Sales"

# one row per order (raw file has one row per order item)
o = df.groupby("Order Id").agg(
    date=(odate, "first"), real=(real, "max"), sched=(sched, "max"),
    late=(late, "max"), mode=(mode, "first"), region=(region, "first"),
    status=("Delivery Status", "first"), qty=(qty, "sum"), sales=(sales, "sum")
).reset_index()
o["date"] = pd.to_datetime(o["date"])
o["delay_days"] = o["real"] - o["sched"]
print("Orders:", len(o), "| Date range:", o["date"].min().date(), "to", o["date"].max().date())

Orders: 65752 | Date range: 2015-01-01 to 2018-01-31


## 2. Late-delivery analysis

In [3]:
print("Total orders:", len(o))
print("Late orders:", int(o.late.sum()), "| Late %:", round(o.late.mean()*100, 1))

print("\nLate % by shipping mode:")
print((o.groupby("mode").late.mean()*100).round(1).sort_values(ascending=False))

print("\nTop 5 regions by late %:")
r = (o.groupby("region").late.mean()*100).round(1).sort_values(ascending=False)
print(r.head(5))

lateo = o[o.late == 1]
print("\nAvg delay on late orders (days):", round(lateo.delay_days.mean(), 2))
print("Share of late orders delayed 2+ days: %", round((lateo.delay_days >= 2).mean()*100, 1))

Total orders: 65752
Late orders: 36048 | Late %: 54.8

Late % by shipping mode:
mode
First Class       95.3
Second Class      76.7
Same Day          46.1
Standard Class    38.1
Name: late, dtype: float64

Top 5 regions by late %:
region
Central Africa    57.6
East Africa       56.8
Eastern Europe    56.0
South of  USA     56.0
West Asia         56.0
Name: late, dtype: float64

Avg delay on late orders (days): 1.62
Share of late orders delayed 2+ days: % 41.4


## 3. Exception queue (200 most delayed orders)

In [4]:
# Exception queue: 200 most delayed orders, with priority and suggested escalation owner
exc = lateo.sort_values("delay_days", ascending=False).head(200).copy()
exc["priority"] = np.where(exc.delay_days >= 3, "High", "Medium")
exc["escalate_to"] = "Carrier / " + exc["region"]
exc[["Order Id","date","mode","region","sched","real","delay_days","priority","escalate_to"]].to_csv("exception_queue.csv", index=False)
print("Saved exception_queue.csv with", len(exc), "orders")
exc.head(10)

Saved exception_queue.csv with 200 orders


,Order Id,date,real,sched,late,mode,region,status,qty,sales,delay_days,priority,escalate_to
29,35,2015-01-01 11:54:00,6,2,1,Second Class,Central America,Late delivery,1,129.990005,4,High,Carrier / Central America
65712,77165,2018-01-31 09:59:00,6,2,1,Second Class,Eastern Asia,Late delivery,1,215.820007,4,High,Carrier / Eastern Asia
27253,32620,2016-04-21 03:53:00,6,2,1,Second Class,South of USA,Late delivery,1,129.990005,4,High,Carrier / South of USA
27244,32610,2016-04-21 00:23:00,6,2,1,Second Class,East of USA,Late delivery,18,1239.870026,4,High,Carrier / East of USA
21,25,2015-01-01 08:24:00,6,2,1,Second Class,Central America,Late delivery,1,399.980011,4,High,Carrier / Central America
53065,63660,2017-07-18 06:34:00,6,2,1,Second Class,Southern Europe,Late delivery,2,239.980007,4,High,Carrier / Southern Europe
9099,10905,2015-06-09 04:09:00,6,2,1,Second Class,Southern Europe,Late delivery,10,989.910019,4,High,Carrier / Southern Europe
9107,10915,2015-06-09 07:39:00,6,2,1,Second Class,Northern Europe,Late delivery,5,749.960022,4,High,Carrier / Northern Europe
9115,10925,2015-06-09 11:09:00,6,2,1,Second Class,Western Europe,Late delivery,1,299.980011,4,High,Carrier / Western Europe
9120,10930,2015-06-09 12:54:00,6,2,1,Second Class,Southern Europe,Late delivery,4,499.960007,4,High,Carrier / Southern Europe
